In [ ]:
excel_batchQname = "combined_raw_uni_prot_R.xlsx"
first_row = 16
PTMs = "PTMs_info_to_use_formatted.csv"
Abeta_peptides = "Abeta_description.csv"
Diagnosis_col = "Diagnosis" ##change to represent column
meta_file = "meta.csv" ##columns should be 1.BatchName 2.CaseID 3.Diagnosis 4.Sample#
custom_order = ['Ctl', 'AD', 'VAD'] ## Anything you don't want such as IR need to have name "Exclude"
corr_data = "Celltype_corrected_rlm_beta_less1_BannerSunVaDcohort.csv" # need to be proteins in columns
output_name = 'Supplement_table_VADv0.0.0.xlsx'
Tau_info = "Tau_renaming.csv"

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load JUMP batch-q result
df= pd.read_excel(excel_batchQname, skiprows = (first_row-1))
df=df.set_index("ProteinAccession")

In [ ]:
print("JUMP output table loaded successfully.")

In [ ]:
# Extract Proteins info
print("Extracting Proteins info from JUMP output table.")

In [ ]:
protein_info = df.iloc[:,1:3]
protein_info["PSMs"] = df.loc[:,df.columns.str.contains("PSM")].replace(0, np.nan).mean(axis=1).round().astype(int)
protein_info["TotalPeptides"] = df.loc[:,df.columns.str.contains("TotalPeptides")].replace(0, np.nan).mean(axis=1).round().replace(np.nan,0).astype(int)
protein_info["UniquePeptides"] = df.loc[:,df.columns.str.contains("UniquePeptides")].replace(0, np.nan).mean(axis=1).round().replace(np.nan,0).astype(int)
protein_info["Coverage"] = df.loc[:,df.columns.str.contains("Coverage")].replace(0, np.nan).mean(axis=1).round(2)

In [ ]:
protein_info["Identifier"] = protein_info.index.str.replace(r'[^a-zA-Z0-9_]', '.', regex=True)
protein_info = protein_info.reset_index().set_index("Identifier")
protein_info["Category"] = "Protein"
protein_info.to_csv("Protein_info.csv")

In [ ]:
print("Successfully extracted protein information.")
print("PSMs,TotalPeptides,UniquePeptides and Coverage as based on the average across all the batches.")
print("Total proteins from JUMP output table: ", len(protein_info))

In [ ]:
# add PTMs info in addition to Proteins info

In [ ]:
# ##Will add manually
# PTM = pd.read_csv("PTMs_info_to_use.csv")
# PTM["identifier"] = PTM.Peptides.str.replace(r'[^a-zA-Z0-9_]', '.', regex=True)
# PTM.to_csv("PTMs_info_to_use_formatted.csv")

In [ ]:
print("Loading PTM information and adding along with proteins information.")

In [ ]:
PTMs = pd.read_csv(PTMs, index_col=0)
Peptides = pd.read_csv(Abeta_peptides, index_col=0)
PTMs = pd.concat([Peptides,PTMs])
identifier_info = pd.concat([protein_info,PTMs])
identifier_info.to_csv("identifier_info.csv")

In [ ]:
identifier_info = identifier_info[['Category', 'ProteinAccession', 'ProteinDescription', 'GN', 'PSMs', 'TotalPeptides',
       'UniquePeptides', 'Coverage']]

In [ ]:
#Load metadata and sort the diagnosis as desired

In [ ]:
print("Meta file loading")

In [ ]:
meta = pd.read_csv(meta_file, index_col=0)
meta.dropna(subset=[Diagnosis_col], inplace=True)
meta = meta[meta[Diagnosis_col] != "Exclude"]

In [ ]:
meta[Diagnosis_col] = pd.Categorical(meta[Diagnosis_col], categories=custom_order, ordered=True)
meta = meta.sort_values(Diagnosis_col)
meta.to_csv("meta_order_used.csv")

In [ ]:
Diagnosis_count = meta[Diagnosis_col].value_counts().reset_index()
total_count = Diagnosis_count['count'].sum()
total_row = pd.DataFrame({'Diagnosis': ['Total'], 'count': [total_count]})
Diagnosis_count = pd.concat([Diagnosis_count, total_row], ignore_index=True)

In [ ]:
print("Meta loaded successfully and here are the diagnosis count \n", Diagnosis_count)

In [ ]:
#load corrected result
print("Normalized and corrected result are being loaded")

In [ ]:
corrected_data = pd.read_csv(corr_data, index_col=0)
corrected_data = (2**corrected_data).round()
corrected_data = corrected_data.loc[:,~corrected_data.columns.str.contains(r"co\.|cu\.", case=False, regex=True)]

In [ ]:
corrected_data = meta.merge(corrected_data, left_index=True, right_index=True, how="inner")
corrected_data = corrected_data.transpose()

In [ ]:
print(corrected_data.shape[1], "Samples are retained from normalized and corrected result.", )

In [ ]:
rows_above_header = corrected_data.iloc[:3]

In [ ]:
corrected_data.columns = corrected_data.iloc[2]
corrected_data = corrected_data.drop(corrected_data.index[0:meta.shape[1]])

In [ ]:
print("Creating a compiled excel table")
print("Contaminants and custome proteins are filtered.")

In [ ]:
data = identifier_info.merge(corrected_data, left_index=True, right_index=True, how="right")
data = data.loc[~data['GN'].astype(str).str.contains(r"^CON_", case=False, regex=True)]
data = data.loc[~data['ProteinAccession'].astype(str).str.contains(r"co\.|cu\.", case=False, regex=True)]

In [ ]:
data.rename(columns={"ProteinAccession":"Protein accession", "ProteinDescription": "Description"
                     , "GN": "Gene name", "TotalPeptides": "Total peptides"
                    , "UniquePeptides": "Unique peptides"}, inplace=True)

In [ ]:
print("PTMs with less than 50% coverage at protein level are filtered.")

In [ ]:
data = data.loc[~((data["Category"].str.contains("PTM_")) & (data["Coverage"] < 50))]
data.loc[data["Category"].str.contains("PTM_", na=False), "Coverage"] = np.nan
data.loc[data["Category"].str.contains("PTM_", na=False), "PSMs"] = np.nan

In [ ]:
count_result = pd.DataFrame(index = data.columns)
count_result["Count"] = data.count()

In [ ]:
data = data.replace(np.nan, "NA")
data = data.rename_axis("Unique identifier")

In [ ]:
print("Tables are sorted for category and within each category sorted by gene names.")

In [ ]:
custom_cateory = ['Peptide', 'Protein', 'PTM_phosphorylation', 'PTM_ubiquitination']
data["Category"] = pd.Categorical(data["Category"], categories=custom_cateory, ordered=True)
data = data.sort_values(by=["Category", "Gene name"])

In [ ]:
tau_rename = pd.read_csv(Tau_info, index_col=0)

In [ ]:
for index in data.index:
    if index in tau_rename.index:
        # Update the "Protein accession" if there's a match
        data.loc[index, "Protein accession"] = tau_rename.loc[index, "ProteinAccession"]
    else:
        continue

In [ ]:
print("Generating QC data i.e. Average, median, min and max.")

In [ ]:
data_QC = data.applymap(lambda x: pd.to_numeric(x, errors='coerce'))

In [ ]:
QC = pd.DataFrame(index = data_QC.columns)
QC["Mean"] = data_QC.mean(axis=0)
QC["Median"] = data_QC.median(axis=0)
QC["Max"] = data_QC.max(axis=0)
QC["Min"] = data_QC.min(axis=0)
QC = QC.round(1)

In [ ]:
rows_above_header = rows_above_header.transpose().reset_index().rename(columns={"index": "BatchNames"}).set_index("Sample#")
QC = count_result.merge(QC, left_index=True, right_index=True, how="left").merge(rows_above_header, left_index=True, right_index=True, how="left").transpose()

In [ ]:
print("Downloading the result........")

In [ ]:
from openpyxl import load_workbook

# Create a new Excel writer object
with pd.ExcelWriter(output_name, engine='openpyxl') as writer:
    # Create a DataFrame to hold the header text
    header_data = pd.DataFrame({
        0: ["Supplementary Table X. Profiling whole proteome and unenriched PTMs in brain tissue from AD cohort 1",
            "Postmortem brain tissues were collected and analyzed by TMT-LC/LC-MS/MS, followed by JUMP and JUMPptm analyses. After identification and quantification, contaminated proteins were removed. TMT reporter ion intensities were normalized, and covariates (e.g., batch, age, sex, and PMI) were corrected, along with cell-type correction for each protein. 'NA' indicates data not available. 'Unique identifier' was used for computation. 'Category' indicates the analysis of peptides, proteins, or PTM peptides. 'Protein accession' was sourced from the database, with detailed 'Description'. 'Gene name' was extracted from the Uniprot database. 'PSMs' refer to the total number of peptide-spectrum matches for each protein averaged from all batches. 'Total peptides' and 'Unique peptides' represent the total or unique number of peptides identified for each protein. 'Coverage' indicates the percentage of the amino acid sequence in a protein that was sequenced by MS, averaged from all batches. Brain samples were listed by disease types and numeric ID numbers. TMT reporter ion intensities were summarized to represent peptide/protein abundance. In PTM data, phosphorylation and ubiquitination at specific residues were identified by a mass shift of 79.97 Da or 114.04 Da in the peptide sequences. 'X' in the peptide sequence indicates its location at the N-terminus of proteins, and an extra dot '.' in the peptide sequence indicates its location at the C-terminus of proteins."
    ]
})

    # Write the header_data as the first two rows
    header_data.to_excel(writer, sheet_name='Sheet1', index=False, header=False)
    worksheet = writer.sheets['Sheet1']
    last_row = len(header_data)  # header is 2 rows
    worksheet.insert_rows(last_row + 1, amount=1)  # 2 empty row

    # Write Diagnosis_count below the empty row
    Diagnosis_count.to_excel(writer, sheet_name='Sheet1', startrow=last_row + 1, index=False)
    diagnosis_last_row = last_row + len(Diagnosis_count) + 2
    worksheet.insert_rows(diagnosis_last_row + 1, amount=1)
    
     # Write QC DataFrame
    QC.to_excel(writer, sheet_name='Sheet1',startrow=diagnosis_last_row + 1, index=True, header=False)
    QC_last_row = diagnosis_last_row + len(QC) + 1
    
    # Write data DataFrame starting below the empty row
    data.to_excel(writer, sheet_name='Sheet1', startrow=QC_last_row, index=True, header=True)
    
from openpyxl.styles import Font, Alignment
start_formate = identifier_info.shape[1]+2
start_formate_row = QC_last_row -7 

# Now load the workbook to apply styles
wb = load_workbook(output_name)
ws = wb['Sheet1']

# Insert an empty column at the first position
ws.insert_cols(1)

# Set font and alignment for all cells
for row in ws.iter_rows():
    for cell in row:
        cell.font = Font(name='Arial', size=11)
        cell.alignment = Alignment(horizontal='left')
        
        if cell.column > start_formate and cell.row > start_formate_row:
            # Column index starts at 1, so column 11 is 'K'
            if isinstance(cell.value, (int, float)):
                if cell.value > 10:
                    # Format the cell in scientific notation
                    cell.number_format = '0.0E+00'  # Scientific notation
                else:
                    # Format the cell to 2 decimal places
                    cell.number_format = '0.00'
                    

# Save the workbook
wb.save(output_name)

In [ ]:
print("Download complete")